# Модуль 7. Архитектура игры: ECS, MVC, слоистая архитектура, async/threading

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр
**Модуль:** 7 — Архитектура игры
**Теория:** 8 ч | **Практика:** 10 ч | **Самостоятельная работа:** 8 ч

---

## О чём этот модуль

До сих пор мы строили игру «снизу вверх»: классы, системы, паттерны. Модуль 7 поднимает уровень абстракции — мы смотрим на игру **сверху**: как организовать весь код, чтобы он не превратился в клубок.

Четыре вопроса этого модуля:

1. **Как хранить данные сущностей?** — ECS (Entity-Component-System) вместо толстых классов.
2. **Как разделить логику и представление?** — MVC/MVVM.
3. **Как разложить код по слоям?** — слоистая архитектура (domain, application, infrastructure, presentation).
4. **Как делать несколько дел одновременно?** — базовые threading и asyncio.

## Где мы находимся в проекте

После модуля 6 проект «Тайна исчезнувшего дива» имеет:

| Слой | Что есть |
|---|---|
| **Сущности** | `Detective`, `Div`, `Location`, `Clue` + компоненты |
| **Системы** | Командные и обновления |
| **Паттерны** | State, Strategy, Command, Observer, Factory, Singleton |
| **Оркестрация** | `Game`, `GameLoop`, `WorldState`, `EventBus` |
| **Инфраструктура** | `UI`, `InputHandler`, `Renderer`, `AudioManager`, `Config` |

**Чего не хватает:** сущности всё ещё классы-«толстяки», код в одной куче, загрузка ассетов блокирует игру.

## Что вы изучите

1. ECS: Entity, Component, System и как они взаимодействуют.
2. Сравнение ECS и классического ООП.
3. Простую ECS на Python.
4. MVC/MVVM в играх.
5. Слоистую архитектуру: domain, application, infrastructure, presentation.
6. Базовые threading: Lock, Queue.
7. Базовые asyncio: async/await, Task.
8. Сценарии использования и риски async/threading.
9. Интеграцию в проект.

## Содержание
1. Что такое ECS
2. ECS vs классическое ООП
3. Простая ECS на Python
4. MVC/MVVM в играх
5. Слоистая архитектура
6. Базовые threading
7. Базовые asyncio
8. Сценарии и риски
9. Итоги и литература

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

import threading, asyncio
print("threading и asyncio доступны")

---
# 1. Что такое ECS

## 1.1. Три сущности

**ECS** (Entity-Component-System) — архитектурный паттерн, который разделяет данные и логику:

| Понятие | Что это | Пример |
|---|---|---|
| **Entity** | Просто идентификатор | «сущность 42» |
| **Component** | Чистые данные, без логики | `Position(x, y)`, `Health(hp)` |
| **System** | Логика над компонентами | `MovementSystem`, `RenderSystem` |

## 1.2. Как это работает

Сущность — это набор компонентов. Система выбирает сущности с нужными компонентами и обрабатывает их.

```
Entity #1: [Position, Velocity, Health]
Entity #2: [Position, Drawable]
Entity #3: [Health]
```

`MovementSystem` обрабатывает только те сущности, у которых есть `Position` и `Velocity`. `RenderSystem` — те, у которых есть `Position` и `Drawable`.

## 1.3. Ключевая идея

**Данные и логика разделены.** Компоненты не знают о системах, системы не знают о конкретных сущностях — только о компонентах. Это максимум композиции и минимум связанности.

## 1.4. Как это улучшит проект

**До:** `Detective` и `Div` — классы с методами и полями вперемешку.

**После:** детектив — сущность с компонентами `Position`, `Health`, `Inventory`. Его поведение — в системах. Новое поведение = новая система, без правки сущностей.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Component](https://gameprogrammingpatterns.com/component.html).
- [Wikipedia: Entity Component System](https://en.wikipedia.org/wiki/Entity_component_system).
- [Sander Mertens: ECS FAQ](https://github.com/SanderMertens/ecs-faq).

---
# 2. ECS vs классическое ООП

## 2.1. Разница подходов

| Критерий | Классическое ООП | ECS |
|---|---|---|
| Данные и логика | В одном классе | Разделены |
| Наследование | Глубокая иерархия | Почти нет |
| Композиция | Через атрибуты | Через компоненты |
| Расширение | Новый подкласс | Новый компонент/система |
| Кэш-эффективность | Низкая | Высокая (данные вместе) |

## 2.2. Когда ECS, а когда ООП

**ECS хорош для:**
- игр с тысячами похожих объектов (пули, частицы, юниты);
- данных, которые удобно обрабатывать пакетами;
- частых изменений поведения.

**ООП хорош для:**
- небольшого числа сложных объектов;
- UI, инструментов, редакторов;
- проектов, где важна наглядность.

## 2.3. Промежуточный вариант

Полноценный ECS с кэш-эффективностью (archetype ECS) в Python не обязателен. Для учебного проекта достаточно **лёгкого ECS**: сущность хранит dict компонентов, системы перебирают сущности. Скорость не критична — важна архитектура.

## Как это улучшит проект

В «Тайне исчезнувшего дива» сущностей немного, поэтому мы внедрим **лёгкий ECS**, чтобы развязать данные и логику, но не станем городить полноценный движок.

## Для углублённого изучения

- [Wikipedia: Entity Component System](https://en.wikipedia.org/wiki/Entity_component_system).
- [Sander Mertens: ECS FAQ](https://github.com/SanderMertens/ecs-faq).
- Nystrom R. «Game Programming Patterns»: [Data Locality](https://gameprogrammingpatterns.com/data-locality.html).

---
# 3. Простая ECS на Python

## 3.1. Компоненты — dataclass

```python
@dataclass
class Position:
    x: float = 0.0
    y: float = 0.0

@dataclass
class Velocity:
    dx: float = 0.0
    dy: float = 0.0

@dataclass
class Health:
    hp: int = 100
```

## 3.2. Сущность — идентификатор

```python
class Entity:
    _next_id = 0
    def __init__(self):
        self.id = Entity._next_id
        Entity._next_id += 1
        self.components = {}
```

## 3.3. Система — логика

```python
class MovementSystem:
    def update(self, entities, dt):
        for e in entities:
            if "position" in e.components and "velocity" in e.components:
                pos = e.components["position"]
                vel = e.components["velocity"]
                pos.x += vel.dx * dt
                pos.y += vel.dy * dt
```

## 3.4. Сборка сущности

```python
detective = Entity()
detective.components["position"] = Position(0, 0)
detective.components["health"] = Health(100)
```

## 3.5. Как это улучшит проект

`MovementSystem` больше не знает про класс `Detective` — только про компоненты `Position` и `Velocity`. Добавить движущийся объект = добавить ему два компонента.

## Для углублённого изучения

- [esper](https://pypi.org/project/esper/) — компактная ECS-библиотека для Python.
- [Sander Mertens: ECS FAQ](https://github.com/SanderMertens/ecs-faq).

---
# 4. MVC/MVVM в играх

## 4.1. MVC

**MVC** (Model-View-Controller) разделяет три ответственности:

| Часть | Что делает | Пример |
|---|---|---|
| **Model** | Данные и бизнес-логика | `WorldState`, сущности |
| **View** | Отображение | `Renderer`, `UI` |
| **Controller** | Ввод и реакции | `InputHandler`, команды |

**Правило зависимостей:** Model не знает про View и Controller. View читает Model. Controller изменяет Model.

## 4.2. MVVM

**MVVM** (Model-View-ViewModel) — вариант MVC, где ViewModel готовит данные для View (форматирование, привязки). В играх применяется в UI (инвентарь, панели), реже — в игровой логике.

## 4.3. MVC в игре

В «Тайне исчезнувшего дива»:
- **Model** — `WorldState`, компоненты, системы.
- **View** — `Renderer` (PyGame-отрисовка), `UI`.
- **Controller** — `InputHandler` + `Command`.

События через `EventBus` (модуль 6) связывают слои, не связывая их напрямую.

## Как это улучшит проект

**До:** `UI` читает и меняет `WorldState` напрямую.

**После:** `InputHandler` формирует команды, `WorldState` меняется через них, `Renderer` только читает и рисует.

## Для углублённого изучения

- [Wikipedia: Model–view–controller](https://en.wikipedia.org/wiki/Model%E2%80%93view%E2%80%93controller).
- Martin R. C. «Clean Architecture», глава про представления.

---
# 5. Слоистая архитектура

## 5.1. Зачем слои

Если весь код в одной папке — любые два модуля могут зависеть друг от друга, и со временем код превращается в клубок. **Слои** ограничивают направления зависимостей.

## 5.2. Классические слои

| Слой | Что содержит | Пример |
|---|---|---|
| **domain** | Чистая логика, правила игры | `Entity`, компоненты, `WorldState` |
| **application** | Сценарии использования | `Game`, `GameLoop`, системы |
| **infrastructure** | Техника: файлы, звук, сеть | `AudioManager`, `Config`, загрузчики |
| **presentation** | Отображение и ввод | `Renderer`, `UI`, `InputHandler` |

## 5.3. Правило зависимостей

Зависимости идут **внутрь**: presentation → application → domain. Domain ни от кого не зависит. Infrastructure реализует интерфейсы, определённые в domain/application.

```
presentation → application → domain
     ↑                            ↑
     └──────── infrastructure ────┘
```

## 5.4. Как это улучшит проект

Сейчас проект лежит в `core/`, `game/`, `entities/`. После модуля 7 он будет разложен по слоям: domain не знает про PyGame и файлы, presentation не знает про правила игры.

## Для углублённого изучения

- Martin R. C. «Clean Architecture», глава про слои и границы.
- [Refactoring Guru: Layered Architecture](https://refactoring.guru/design-patterns/layered-architecture).

---
# 6. Базовые threading

## 6.1. Зачем потоки

Потоки позволяют выполнять несколько задач «одновременно». В играх это полезно для:
- фоновой загрузки ассетов;
- проигрывания музыки (в отдельном потоке);
- лёгкого логирования.

**Но:** Python имеет GIL, поэтому потоки не ускоряют CPU-вычисления. Они полезны для I/O (файлы, сеть, звук).

## 6.2. threading.Thread

```python
import threading

def background_task(stop_event):
    while not stop_event.is_set():
        do_work()

thread = threading.Thread(target=background_task, args=(stop_event,), daemon=True)
thread.start()
```

`daemon=True` — поток завершится вместе с основной программой.

## 6.3. Lock — защита общих данных

Если два потока пишут в один объект, нужна блокировка:

```python
lock = threading.Lock()

with lock:
    shared_counter += 1
```

Без `Lock` возможна гонка: два потока прочитают одно значение, оба увеличат, оба запишут — один инкремент потеряется.

## 6.4. Queue — передача данных между потоками

Очередь потокобезопасна — её удобно использовать для событий между потоками:

```python
import queue
q = queue.Queue()
q.put(item)        # в потоке-производителе
item = q.get()     # в потоке-потребителе
```

## Как это улучшит проект

Загрузка ассетов в отдельном потоке не заблокирует игровой цикл. `Queue` передаст загруженные ассеты в основной поток.

## Для углублённого изучения

- Документация Python: [threading](https://docs.python.org/3/library/threading.html), [queue](https://docs.python.org/3/library/queue.html).
- [Real Python: Threading in Python](https://realpython.com/intro-to-python-threading/).

---
# 7. Базовые asyncio

## 7.1. Чем asyncio отличается от потоков

**asyncio** — однопоточная кооперативная многозадачность. Код сам «уступает» выполнение на `await`. Полезно для I/O без накладных расходов потоков.

```python
import asyncio

async def load_asset(path):
    await asyncio.sleep(0.1)  # имитация I/O
    return f"asset {path}"

async def main():
    results = await asyncio.gather(
        load_asset("a.png"),
        load_asset("b.png"),
    )
    print(results)

asyncio.run(main())
```

## 7.2. Ключевые понятия

| Понятие | Что это |
|---|---|
| `async def` | Корoutine-функция |
| `await` | Уступает выполнение |
| `asyncio.gather` | Запускает несколько задач параллельно |
| `asyncio.run` | Запускает цикл событий |

## 7.3. asyncio vs threading

| Критерий | threading | asyncio |
|---|---|---|
| Параллелизм | Настоящий (I/O) | Кооперативный |
| GIL | Ограничивает CPU | Не мешает I/O |
| Сложность | Гонки, Lock | Проще, нет гонок |
| Сценарий | Звук, блокирующий I/O | Сетевые запросы, много I/O |

## Как это улучшит проект

`asyncio.gather` параллельно загрузит несколько ассетов. Для простых сценариев это чище, чем потоки.

## Для углублённого изучения

- Документация Python: [asyncio](https://docs.python.org/3/library/asyncio.html).
- [Real Python: Async IO in Python](https://realpython.com/async-io-python/).

---
# 8. Сценарии и риски async/threading

## 8.1. Типичные сценарии в играх

| Сценарий | Инструмент |
|---|---|
| Загрузка ассетов при старте | threading или asyncio |
| Фоновая музыка | threading (отдельный поток) |
| Логирование | threading + Queue |
| Сетевые заглушки (лидерборды) | asyncio |
| Тяжёлый расчёт (ИИ, физика) | НЕ поток (GIL) — разбивать на кадры |

## 8.2. Риски

1. **Гонки данных.** Общий объект без `Lock` — источник трудноуловимых багов.
2. **Дедлоки.** Два потока ждут друг друга.
3. **Недетерминизм.** Порядок выполнения зависит от планировщика.
4. **Смешивание потоков и PyGame.** PyGame не потокобезопасен: рисовать и менять состояние нужно в основном потоке.

## 8.3. Правила безопасности

- Изменяйте игровое состояние **только в основном потоке**.
- Потоки — только для I/O (загрузка, звук, сеть).
- Передавайте данные через `Queue`, не через общие переменные.
- `daemon=True` для фоновых потоков.

## Как это улучшит проект

Мы загрузим ассеты асинхронно, а фоновую музыку/логирование вынесем в поток. Игровой цикл останется в основном потоке — безопасно и предсказуемо.

## Для углублённого изучения

- Документация Python: [threading](https://docs.python.org/3/library/threading.html), [asyncio](https://docs.python.org/3/library/asyncio.html), [queue](https://docs.python.org/3/library/queue.html).
- [Real Python: Threading in Python](https://realpython.com/intro-to-python-threading/).

---
# 9. Интеграция в проект

## 9.1. Что появится

| Компонент | Что это |
|---|---|
| `ecs.py` | Entity, компоненты, системы |
| Слои | `domain/`, `application/`, `infrastructure/`, `presentation/` |
| `AssetLoader` | Асинхронная загрузка ассетов |
| `BackgroundTask` | Поток для музыки/логирования |

## 9.2. Как это подготовит модуль 8

- Чистые слои и ECS упростят тестирование (domain без I/O легко тестировать).
- Разделённые зависимости упростят рефакторинг и код-ревью.
- Отдельные модули — понятная структура для защиты.

## Литература

**Базовая:**
- Nystrom R. «Game Programming Patterns»: [Component](https://gameprogrammingpatterns.com/component.html), [Data Locality](https://gameprogrammingpatterns.com/data-locality.html).
- Martin R. C. «Clean Architecture».

**Углублённая:**
- [Wikipedia: Entity Component System](https://en.wikipedia.org/wiki/Entity_component_system).
- [Sander Mertens: ECS FAQ](https://github.com/SanderMertens/ecs-faq).
- Документация Python: [threading](https://docs.python.org/3/library/threading.html), [asyncio](https://docs.python.org/3/library/asyncio.html).

## Что дальше

В практической работе вы спроектируете ECS, разложите проект по слоям и добавите асинхронную загрузку и фоновый поток.